# RSA Encryption and Decryption

With the keys from the key generation notebook, encryption and decryption is surprisingly simple. Both are just a single modular exponentiation.

## Setup

Reusing the keys from the key generation notebook:

In [1]:
using Primes

p, q    = 61, 53
n       = p * q
phi_n   = (p - 1) * (q - 1)
e       = 7
d       = invmod(e, phi_n)

println("Public key:  (e = $e, n = $n)")
println("Private key: (d = $d, n = $n)")

Public key:  (e = 7, n = 3233)
Private key: (d = 1783, n = 3233)


## Encryption

Given a message $m$ (as an integer, where $m \lt n$), encryption is:

$$c = m^e \pmod{n}$$

The message is raised to the public exponent and reduced mod $n$.

The output is the ciphertext $c$.

In [2]:
m = 42 # message must be < n 

c = powermod(m, e, n)

println("Message:    $m")
println("Ciphertext: $c")

Message:    42
Ciphertext: 240


$42$ becomes $240$. Without knowing $d$, recovering $m$ from $c$ requires solving the **discrete logarithm problem**, which is effectively impossible for large $n$.

## Decryption

Given ciphertext $c$, decryption is:

$$m = c^d \pmod{n}$$

The private exponent $d$ undoes what $e$ did.

In [3]:
m_decrypted = powermod(c, d, n)

println("Ciphertext:          $c")
println("Decrypted message:   $m_decrypted")
println("Matches original:    $(m_decrypted == m)")

Ciphertext:          240
Decrypted message:   42
Matches original:    true


In [ ]:
using CairoMakie

# Encrypt all possible messages with our RSA parameters
plaintexts = 0:n-1
ciphertexts = [powermod(m, e, n) for m in plaintexts]

# Create scatter plot
fig = Figure(size=(700, 600))
ax = Axis(fig[1, 1], xlabel="Plaintext m", ylabel="Ciphertext c = m^e mod n", 
          title="RSA Encryption: A Permutation of the Message Space")

scatter!(ax, plaintexts, ciphertexts, markersize=60, color=(:steelblue, 0.6), strokewidth=1, strokecolor=:black)

# Add a few labeled points for clarity
for m in [0, 1, 42]
    c = powermod(m, e, n)
    text!(ax, m, c + 50, text="$m→$c", align=(:center, :bottom), fontsize=8)
end

xlims!(ax, -100, n+100)
ylims!(ax, -100, n+100)

fig

## Visualization: Encryption as a Permutation

RSA encryption is a bijection (permutation) on the message space. Every plaintext maps to exactly one ciphertext:

## Why Does This Work?

This is essentially just Euler's Theorem.

Recall that $d$ is chosen so that $ed \equiv 1 \pmod{\phi(n)}$. This means that $ed = 1 + k\phi(n)$ for some integer $k$. So:

$$m^{ed} = m^{1 + k\phi(n)} = m \cdot (m^{\phi(n)})^k$$

By Euler's Theorem, $m^{\phi(n)} \equiv 1 \pmod{n}$ when $\gcd(m, n) = 1$. So:

$$m^{ed} \equiv m \cdot 1^k \equiv m \pmod{n}$$

Encrypting with $e$ then decrypting with $d$ gives back $m$. This is because $e$ and $d$ are modular inverses of each other in the exponent.

## Limitations

The implementation I've written here is intentionally limited. RSA in practice is much more complex in two important ways: Padding and Small Public Exponents.

Encrypting the same message twice with the same key always produces the same ciphertext. This is a very serious problem. An attacker can build a dictionary of known plaintexts and ciphertexts. Real RSA uses padding schemes like OAEP to randomize encryption.

Using a small $e$, like $7$ in this example, without padding opens the door to attacks. If the same message is encrypted to several recipients with the same small $e$, an attacker can recover the message using the **Chinese Remainder Theorem**. This is Håstad's broadcast attack which is covered in a separate notebook.